# 03-1. Body 데이터 증강 — 좌우반전

`dataset/processed/body/` 의 원본 이미지마다 좌우반전(horizontal flip) 버전을 생성해 저장합니다.  
파일명 규칙: `{원본 stem}_flip.png`

## 1. 현재 상태 확인

In [ ]:
import os
import random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 노트북 위치와 관계없이 프로젝트 루트 CWD 고정
_root = Path(os.path.abspath(''))
for _p in [_root] + list(_root.parents):
    if (_p / 'dataset' / 'processed').exists():
        os.chdir(_p)
        break
print(f'CWD: {Path.cwd()}')

BODY_DIR = Path('dataset/processed/body')

all_files = list(BODY_DIR.glob('*.png'))
originals = [f for f in all_files if not f.stem.endswith('_flip')]
flipped   = [f for f in all_files if f.stem.endswith('_flip')]

print(f'현재 전체 파일  : {len(all_files):,}개')
print(f'원본            : {len(originals):,}개')
print(f'기존 반전 파일  : {len(flipped):,}개')
print(f'증강 후 예상    : {len(originals) * 2:,}개')

## 2. 샘플 미리보기 — 원본 vs 좌우반전

In [ ]:
random.seed(42)
samples = random.sample(originals, min(8, len(originals)))

fig, axes = plt.subplots(2, len(samples), figsize=(len(samples) * 1.4, 3.2))
axes = np.array(axes).reshape(2, len(samples))
bg   = (0.82, 0.82, 0.82)

for col, path in enumerate(samples):
    img  = Image.open(path).convert('RGBA')
    flip = img.transpose(Image.FLIP_LEFT_RIGHT)

    for row, im in enumerate([img, flip]):
        arr   = np.array(im) / 255.0
        alpha = arr[:, :, 3:4]
        comp  = arr[:, :, :3] * alpha + np.array(bg) * (1 - alpha)
        axes[row, col].imshow(np.clip(comp, 0, 1))
        axes[row, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('Flipped',  fontsize=9)
plt.suptitle('Horizontal Flip Preview — Body Sprites', fontsize=12)
plt.tight_layout()
plt.show()

## 3. 좌우반전 생성 및 저장

In [ ]:
# 이미 반전 파일이 있으면 원본만 대상으로 한정
targets = [f for f in BODY_DIR.glob('*.png') if not f.stem.endswith('_flip')]
print(f'반전 생성 대상: {len(targets):,}개')

def flip_and_save(src_path: Path) -> Path:
    dst_path = src_path.with_stem(src_path.stem + '_flip')
    if dst_path.exists():
        return dst_path
    img = Image.open(src_path).convert('RGBA')
    img.transpose(Image.FLIP_LEFT_RIGHT).save(dst_path)
    return dst_path

# macOS Jupyter multiprocessing 충돌 방지 → ThreadPoolExecutor 사용
errors = []
done   = 0

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(flip_and_save, p): p for p in targets}
    for i, fut in enumerate(as_completed(futures), 1):
        try:
            fut.result()
            done += 1
        except Exception as e:
            errors.append((futures[fut], e))
        if i % 500 == 0 or i == len(targets):
            print(f'  {i:,} / {len(targets):,} 완료')

print(f'\n생성 완료: {done:,}개  오류: {len(errors)}개')
if errors:
    for path, err in errors[:5]:
        print(f'  오류: {path.name} — {err}')

## 4. 결과 확인

In [ ]:
all_after  = list(BODY_DIR.glob('*.png'))
orig_after = [f for f in all_after if not f.stem.endswith('_flip')]
flip_after = [f for f in all_after if f.stem.endswith('_flip')]

print(f'원본         : {len(orig_after):,}개')
print(f'반전         : {len(flip_after):,}개')
print(f'전체 (합계)  : {len(all_after):,}개  (증가: +{len(all_after)-len(originals):,})')

# 비율 시각화
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(['body'], [len(orig_after)], color='#5b8dee', label='Original')
ax.barh(['body'], [len(flip_after)], left=len(orig_after), color='#2a9d8f', label='Flipped')
ax.set_xlabel('Frames')
ax.set_title('Body Dataset — Before vs After Augmentation', fontsize=12)
ax.legend()
for spine in ['top', 'right']: ax.spines[spine].set_visible(False)
ax.text(len(orig_after)/2,        0, f'{len(orig_after):,}', ha='center', va='center', color='white', fontweight='bold')
ax.text(len(orig_after)+len(flip_after)/2, 0, f'{len(flip_after):,}', ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 원본-반전 쌍 무작위 확인
random.seed(0)
check_pairs = random.sample(orig_after, min(6, len(orig_after)))

fig, axes = plt.subplots(2, len(check_pairs), figsize=(len(check_pairs) * 1.4, 3.2))
axes = np.array(axes).reshape(2, len(check_pairs))
bg   = (0.82, 0.82, 0.82)

for col, orig_path in enumerate(check_pairs):
    flip_path = orig_path.with_stem(orig_path.stem + '_flip')
    for row, path in enumerate([orig_path, flip_path]):
        arr   = np.array(Image.open(path).convert('RGBA')) / 255.0
        alpha = arr[:, :, 3:4]
        comp  = arr[:, :, :3] * alpha + np.array(bg) * (1 - alpha)
        axes[row, col].imshow(np.clip(comp, 0, 1))
        axes[row, col].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('Saved Flip', fontsize=9)
plt.suptitle('Saved Flip Verification', fontsize=12)
plt.tight_layout()
plt.show()